# Daily Practice — 2026-11-20 — ML Testing & MLOps: Building a Data Contract Test Suite

**Dataset:** [Titanic](https://ggplot2.tidyverse.org/reference/diamonds.html) passenger records — 891 real
passengers from the RMS Titanic (class, sex, age, fare, family size, port of embarkation, survival),
loaded from the `mwaskom/seaborn-data` mirror of the classic Titanic dataset.

## Problem statement

A "survival risk" model is live in production, scoring batches of passenger records as they arrive from
an upstream ingestion pipeline. As the QA/MLOps engineer, you don't get to inspect every incoming batch by
eye — pipelines get refactored, upstream joins start dropping rows, and a renamed or reformatted column
slips through all the time. What you *can* do is stop a bad batch from ever reaching the model, the same
way an API test suite stops a malformed request from reaching a service: with an automated **data
contract** — a set of checks derived from a trusted reference dataset that any new batch must satisfy
before it's scored.

Your job is to build a reusable data contract test suite: derive a contract from a reference ("training")
slice of the data, then write checks that catch schema drift (missing/renamed columns, categories that
didn't exist in training), statistically implausible values (out-of-range numbers), and data quality
regressions (a null-rate spike, a wave of duplicate rows) in any new batch — and produce a structured
report a release pipeline could gate on.

## What you should produce

1. A deterministic 70/30 split of the shuffled dataset into `train_df` (used only to *derive* the
   contract) and `holdout_df` (used to *sample* "incoming" batches), using `RANDOM_STATE`.
2. `build_reference_contract(ref_df, categorical_cols, numeric_cols, iqr_multiplier=4.0,
   min_null_tolerance=0.02, null_safety_factor=2.0)` — for every column, records:
   - the allowed category set (categorical columns), or an IQR-based `[low, high]` fence (numeric
     columns) — with a guard for the degenerate case where a column's IQR is 0 (e.g. `parch`, where most
     passengers travel with 0 parents/children and the fence would otherwise collapse to a single point);
   - a null-rate tolerance of `max(min_null_tolerance, null_safety_factor * observed_null_rate)`, so a
     column that's *already* mostly missing in training (like `age`) gets a wider — but still finite —
     allowance than one that's almost always present (like `embarked`).
3. `validate_batch(batch, contract, feature_cols, max_duplicate_rate=0.05)` — returns a tidy DataFrame
   with one row per `(check, column)`, a `PASS`/`FAIL` `status`, and a human-readable `detail` string,
   covering: column presence, null-rate tolerance, categorical domain, numeric range, and a batch-level
   duplicate-row-rate check (computed only over whichever feature columns are actually present, so a
   batch missing a column doesn't crash the suite).
4. `contract_gate(report)` — reduces a `validate_batch` report to
   `{"passed", "n_checks", "n_failed", "failures"}`.
5. Three "incoming" batches of 100 rows sampled from `holdout_df` (never from `train_df`):
   - `batch_clean` — an untouched sample;
   - `batch_schema_break` — a copy with an unseen `embarked` value injected, a few `age` values set
     negative, and the `fare` column dropped entirely (simulating an upstream pipeline change);
   - `batch_null_spike` — a copy with ~70% of `age` set to null (simulating an upstream join starting to
     fail for most rows).

   Run `validate_batch` + `contract_gate` on all three and print each result.
6. A short written verdict (4–6 sentences): for every `FAIL` row across the three batches, say whether
   it's a **hard structural violation** (unambiguous, should always block scoring on its own) or a **soft
   statistical flag** (worth a second look but shouldn't block alone), and justify the split. Then say
   what — if anything — you'd change about the contract so `batch_clean` doesn't trip any soft flags it
   shouldn't.

Try it yourself in the starter cells below before you scroll down to the solution.

## Setup

In [ ]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.width", 120)

## Load the data

In [ ]:
TITANIC_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(TITANIC_URL)

CATEGORICAL_COLS = ["sex", "embarked", "pclass"]
NUMERIC_COLS = ["age", "fare", "sibsp", "parch"]
FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS

df[FEATURE_COLS].info()
df[FEATURE_COLS].head()

## Your task

Fill in the TODOs below. Function signatures are given; the logic is up to you.

In [ ]:
# TODO: shuffle `df` with RANDOM_STATE and split it 70/30 into `train_df` (the reference
# slice used to derive the contract) and `holdout_df` (the pool you'll sample incoming
# batches from). Reset the index on both.

train_df = ...
holdout_df = ...

In [ ]:
def build_reference_contract(ref_df, categorical_cols, numeric_cols, iqr_multiplier=4.0,
                              min_null_tolerance=0.02, null_safety_factor=2.0):
    """Derive a data contract from a reference DataFrame.

    Returns {"columns": {col: spec, ...}} where each spec has:
      - "dtype_kind": "categorical" or "numeric"
      - "null_tolerance": float
      - categorical columns also get "allowed_values": set of observed non-null values
      - numeric columns also get "low" / "high": an IQR*iqr_multiplier fence, falling back
        to the observed min/max when IQR == 0
    """
    # TODO:
    # 1. For each column in categorical_cols + numeric_cols:
    #    a. observed_null_rate = fraction of nulls in ref_df[col]
    #    b. null_tolerance = max(min_null_tolerance, null_safety_factor * observed_null_rate)
    # 2. If the column is categorical: allowed_values = set of its non-null unique values.
    # 3. If the column is numeric: compute Q1/Q3 on the non-null values, then:
    #    - if IQR == 0, use the column's own min/max as [low, high]
    #    - otherwise, low = Q1 - iqr_multiplier * IQR, high = Q3 + iqr_multiplier * IQR
    raise NotImplementedError

In [ ]:
def validate_batch(batch, contract, feature_cols, max_duplicate_rate=0.05):
    """Check `batch` against `contract` and return a tidy report DataFrame with columns
    ["check", "column", "status", "detail"], status in {"PASS", "FAIL"}.

    Checks per feature column: schema_presence, null_rate, then (categorical_domain OR
    numeric_range) depending on the column's dtype_kind. If a column is missing from the
    batch, record the schema_presence failure and skip the rest of that column's checks.
    Finally, one batch-level duplicate_rows check: the fraction of duplicate rows (on
    whichever feature_cols are actually present) must not exceed max_duplicate_rate.
    """
    # TODO:
    # 1. For each column in feature_cols, check presence; if missing, record a FAIL for
    #    "schema_presence" and `continue` to the next column.
    # 2. Otherwise record a PASS for "schema_presence", then check the column's null rate
    #    against contract["columns"][col]["null_tolerance"].
    # 3. For categorical columns, check for values not in "allowed_values".
    #    For numeric columns, check for values outside ["low", "high"].
    # 4. After the column loop, compute the duplicate-row rate on the feature columns that
    #    are actually present in `batch`, and compare it to max_duplicate_rate.
    # 5. Return everything as one DataFrame.
    raise NotImplementedError

In [ ]:
def contract_gate(report):
    """Reduce a validate_batch report to a release-gate decision.

    Returns {"passed": bool, "n_checks": int, "n_failed": int, "failures": DataFrame}
    where "failures" holds only the FAIL rows of `report`.
    """
    # TODO: passed is True iff there are no FAIL rows in `report`.
    raise NotImplementedError

In [ ]:
# TODO:
# 1. Build `contract` by calling build_reference_contract on train_df.
# 2. Build three 100-row batches sampled from holdout_df (use distinct random_state values
#    so the three samples don't overlap in a confusing way):
#    - batch_clean: untouched sample.
#    - batch_schema_break: copy with embarked set to an unseen value "X" for a few rows,
#      age set negative for a few other rows, and the "fare" column dropped entirely.
#    - batch_null_spike: copy with ~70% of "age" set to NaN.
# 3. Run validate_batch + contract_gate on each batch (pass FEATURE_COLS as feature_cols)
#    and print, for each: whether it passed, n_checks, n_failed, and the failures table
#    when it didn't pass.

contract = ...
batch_clean = ...
batch_schema_break = ...
batch_null_spike = ...

### Write-up

TODO: in 4–6 sentences, classify every FAIL row across the three batches as a hard structural
violation or a soft statistical flag, justify the split, and say what you'd change about the
contract so `batch_clean` doesn't trip any soft flags it shouldn't.

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# 1. Deterministic split: train_df derives the contract, holdout_df supplies batches
shuffled = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
split_idx = int(0.7 * len(shuffled))
train_df = shuffled.iloc[:split_idx].reset_index(drop=True)
holdout_df = shuffled.iloc[split_idx:].reset_index(drop=True)

# 2. Contract builder
def build_reference_contract(ref_df, categorical_cols, numeric_cols, iqr_multiplier=4.0,
                              min_null_tolerance=0.02, null_safety_factor=2.0):
    contract = {"columns": {}}
    for col in categorical_cols + numeric_cols:
        observed_null_rate = ref_df[col].isna().mean()
        null_tol = max(min_null_tolerance, null_safety_factor * observed_null_rate)
        entry = {
            "dtype_kind": "categorical" if col in categorical_cols else "numeric",
            "null_tolerance": round(float(null_tol), 4),
        }
        if col in categorical_cols:
            entry["allowed_values"] = set(ref_df[col].dropna().unique().tolist())
        else:
            q1, q3 = ref_df[col].quantile(0.25), ref_df[col].quantile(0.75)
            iqr = q3 - q1
            if iqr == 0:
                entry["low"] = float(ref_df[col].min())
                entry["high"] = float(ref_df[col].max())
            else:
                entry["low"] = float(q1 - iqr_multiplier * iqr)
                entry["high"] = float(q3 + iqr_multiplier * iqr)
        contract["columns"][col] = entry
    return contract

# 3. Batch validator
def validate_batch(batch, contract, feature_cols, max_duplicate_rate=0.05):
    rows = []
    present_cols = [c for c in feature_cols if c in batch.columns]

    def add(check, column, status, detail):
        rows.append({"check": check, "column": column, "status": status, "detail": detail})

    for col in feature_cols:
        if col not in batch.columns:
            add("schema_presence", col, "FAIL", "column missing from batch")
            continue
        add("schema_presence", col, "PASS", "column present")

        spec = contract["columns"][col]
        series = batch[col]

        null_rate = series.isna().mean()
        if null_rate > spec["null_tolerance"]:
            add("null_rate", col, "FAIL",
                f"null_rate={null_rate:.3f} exceeds tolerance={spec['null_tolerance']}")
        else:
            add("null_rate", col, "PASS", f"null_rate={null_rate:.3f}")

        non_null = series.dropna()
        if spec["dtype_kind"] == "categorical":
            unseen = set(non_null.unique().tolist()) - spec["allowed_values"]
            if unseen:
                add("categorical_domain", col, "FAIL", f"unseen categories: {sorted(unseen, key=str)}")
            else:
                add("categorical_domain", col, "PASS", "all categories known")
        else:
            out_of_range = ((non_null < spec["low"]) | (non_null > spec["high"])).sum()
            if out_of_range > 0:
                add("numeric_range", col, "FAIL",
                    f"{out_of_range} value(s) outside [{spec['low']:.2f}, {spec['high']:.2f}]")
            else:
                add("numeric_range", col, "PASS",
                    f"all values within [{spec['low']:.2f}, {spec['high']:.2f}]")

    dup_rate = batch.duplicated(subset=present_cols).mean()
    if dup_rate > max_duplicate_rate:
        add("duplicate_rows", "<row>", "FAIL",
            f"duplicate_rate={dup_rate:.3f} exceeds tolerance={max_duplicate_rate}")
    else:
        add("duplicate_rows", "<row>", "PASS", f"duplicate_rate={dup_rate:.3f}")

    return pd.DataFrame(rows)

# 4. Gate
def contract_gate(report):
    failures = report[report["status"] == "FAIL"].reset_index(drop=True)
    return {"passed": failures.empty, "n_checks": len(report), "n_failed": len(failures), "failures": failures}

# 5. Contract + three incoming batches
contract = build_reference_contract(train_df, CATEGORICAL_COLS, NUMERIC_COLS)

batch_clean = holdout_df.sample(n=100, random_state=1).reset_index(drop=True)

batch_schema_break = holdout_df.sample(n=100, random_state=2).reset_index(drop=True).copy()
batch_schema_break.loc[0:4, "embarked"] = "X"
batch_schema_break.loc[5:9, "age"] = -1
batch_schema_break = batch_schema_break.drop(columns=["fare"])

batch_null_spike = holdout_df.sample(n=100, random_state=3).reset_index(drop=True).copy()
null_idx = rng.choice(batch_null_spike.index, size=70, replace=False)
batch_null_spike.loc[null_idx, "age"] = np.nan

for name, batch in [("clean", batch_clean), ("schema_break", batch_schema_break),
                    ("null_spike", batch_null_spike)]:
    report = validate_batch(batch, contract, FEATURE_COLS)
    gate = contract_gate(report)
    print(f"=== batch: {name} ===")
    print(f"passed={gate['passed']}  n_checks={gate['n_checks']}  n_failed={gate['n_failed']}")
    if not gate["passed"]:
        print(gate["failures"].to_string(index=False))
    print()
```

**Typical output**

```
=== batch: clean ===
passed=False  n_checks=22  n_failed=2
        check column status                             detail
numeric_range   fare   FAIL 6 value(s) outside [-81.52, 119.73]
numeric_range  sibsp   FAIL    2 value(s) outside [-4.00, 5.00]

=== batch: schema_break ===
passed=False  n_checks=20  n_failed=4
             check   column status                                detail
categorical_domain embarked   FAIL              unseen categories: ['X']
   schema_presence     fare   FAIL             column missing from batch
     numeric_range    sibsp   FAIL      1 value(s) outside [-4.00, 5.00]
    duplicate_rows    <row>   FAIL duplicate_rate=0.100 exceeds tolerance=0.05

=== batch: null_spike ===
passed=False  n_checks=22  n_failed=4
         check column status                                detail
     null_rate    age   FAIL    null_rate=0.720 exceeds tolerance=0.4141
 numeric_range   fare   FAIL          7 value(s) outside [-81.52, 119.73]
 numeric_range  sibsp   FAIL           1 value(s) outside [-4.00, 5.00]
duplicate_rows    <row>   FAIL duplicate_rate=0.180 exceeds tolerance=0.05
```

**Explanation**

None of the three batches pass cleanly, but they fail for very different reasons, and telling those
reasons apart is the actual job here. `schema_presence` (a column vanished) and `categorical_domain` (a
value that never existed in training showed up, here `embarked == "X"`) are **hard structural
violations**: there's no statistical nuance to them — a column is either there or it isn't, and a
category is either known or it isn't — so they should block scoring unconditionally, every time. The
`null_rate` failure on `age` in `batch_null_spike` (72% missing against a tolerance of 41%, itself already
widened to accommodate training's ~20% baseline missingness) is also effectively a hard violation: the
tolerance was already generous, and blowing through it by nearly 2x is a real ingestion incident, not
noise. By contrast, `numeric_range` on `fare` and `sibsp` fires on *every single batch, including the
untouched clean one* — these are **soft statistical flags**: `fare` is genuinely heavy-tailed (a handful
of real first-class tickets cost 10x the median) and `sibsp` is a small-integer count column, so any fixed
IQR fence will occasionally catch a few legitimate values no matter how wide it's set; they're worth
a second look (e.g. plotted over time to see if the rate itself is drifting) but shouldn't block a batch
on their own. The `duplicate_rows` flag is a related but distinct case: it only fires on the two broken
batches (10% and 18%), not because passengers are literally duplicated, but because dropping `fare` or
nulling most of `age` removes the columns that were keeping otherwise-similar passengers distinguishable
— a useful secondary signal that something upstream changed, even though the "duplicates" aren't real.
To stop `batch_clean` from tripping avoidable soft flags, the most defensible fix is to raise
`iqr_multiplier` further for `fare` specifically (it has real, legitimate long-tail mass that no other
numeric column here has) or drop a hard range fence on it in favor of a looser, log-scale check — rather
than loosening every numeric column uniformly and losing sensitivity on the ones that aren't skewed.

</details>